In [1]:
import pandas as pd
import numpy as np
import pickle
from pathlib import Path
from sklearn.model_selection import KFold
from sklearn.metrics import f1_score
from sklearn.base import clone

# ==========================================
# 1. SETUP PATH & LOAD DATA LATIH
# ==========================================
root_path = Path.cwd().parent
print("⏳ 1. Memuat Data Latih untuk Cross Validation...")

# Kita gunakan data latih (X_train) agar halal 100% tanpa menyentuh data ujian asli
train_df = pd.read_csv(root_path / "Data/processed/train_selected_features.csv")

selected_features = ['TIPI1', 'TIPI2', 'TIPI3', 'TIPI4', 'TIPI5', 'TIPI6', 'TIPI7', 'TIPI8', 'TIPI9', 'TIPI10', 
                     'VCL1', 'VCL2', 'VCL3', 'VCL4', 'VCL5', 'VCL7', 'VCL8', 'VCL9', 'VCL10', 'VCL11', 'VCL12', 
                     'VCL13', 'VCL14', 'VCL15', 'VCL16', 'education', 'urban', 'gender', 'engnat', 'age', 
                     'religion', 'orientation', 'race', 'voted', 'married']
target_cols = ['risk_depression', 'risk_anxiety', 'risk_stress']

X = train_df[selected_features]
Y = train_df[target_cols]

# ==========================================
# 2. LOAD MODEL TERBAIK & THRESHOLD EMAS
# ==========================================
model_path = root_path / "models/multilabel_xgboost_mfo.pkl"
with open(model_path, "rb") as f:
    best_model = pickle.load(f)

# Menggunakan Threshold hasil optimasi kita sebelumnya
optimal_thresholds = [0.41, 0.37, 0.38] # Depresi, Ansietas, Stres

# ==========================================
# 3. PROSES 5-FOLD CROSS VALIDATION
# ==========================================
print(f"\n🔄 2. Memulai 5-Fold Cross Validation...")
print("-" * 50)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X, Y), 1):
    print(f"   ▶️ Sedang melatih Fold ke-{fold}...")
    
    # Membagi data berdasarkan lipatan
    X_fold_train, X_fold_val = X.iloc[train_idx], X.iloc[val_idx]
    Y_fold_train, Y_fold_val = Y.iloc[train_idx], Y.iloc[val_idx]
    
    # CLONE MODEL: Membuat duplikat arsitektur model terbaikmu (agar tidak numpuk memori)
    cv_model = clone(best_model)
    
    # Melatih ulang di fold ini
    cv_model.fit(X_fold_train, Y_fold_train)
    
    # Memprediksi probabilitas di data validasi fold ini
    y_val_proba = cv_model.predict_proba(X_fold_val)
    
    # Siapkan tempat tebakan
    y_fold_pred = np.zeros_like(Y_fold_val)
    
    # Terapkan threshold emas
    for i in range(len(target_cols)):
        # Ambil probabilitas kelas 1
        prob_positive = y_val_proba[i][:, 1]
        # Tebak 1 jika melebihi threshold emas
        y_fold_pred[:, i] = (prob_positive >= optimal_thresholds[i]).astype(int)
        
    # Hitung F1 Macro untuk fold ini
    fold_macro_f1 = f1_score(Y_fold_val, y_fold_pred, average='macro')
    fold_scores.append(fold_macro_f1)
    
    print(f"      ✅ Selesai! Skor F1 Fold {fold}: {fold_macro_f1:.4f}")

# ==========================================
# 4. HASIL AKHIR CROSS VALIDATION
# ==========================================
print("\n" + "=" * 50)
print("🏆 KESIMPULAN 5-FOLD CROSS VALIDATION 🏆")
print("=" * 50)
print(f"Skor setiap Fold : {[round(score, 4) for score in fold_scores]}")
print(f"Rata-rata (Mean) : **{np.mean(fold_scores):.4f}**")
print(f"Simpangan Baku   : ±{np.std(fold_scores):.4f} (Makin kecil makin stabil)")
print("=" * 50)
print("Penjelasan untuk Dosen: Model ini telah dievaluasi 5 kali berturut-turut pada potongan data yang berbeda, dan terbukti menghasilkan akurasi yang stabil. Modeling tidak hanya dilakukan satu kali.")

⏳ 1. Memuat Data Latih untuk Cross Validation...

🔄 2. Memulai 5-Fold Cross Validation...
--------------------------------------------------
   ▶️ Sedang melatih Fold ke-1...


d:\Amikom\Semester 6\Project Data Mining\Project\venv\lib\site-packages\xgboost\core.py:158: UserWarning: [16:04:57] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
d:\Amikom\Semester 6\Project Data Mining\Project\venv\lib\site-packages\xgboost\core.py:158: UserWarning: [16:04:59] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
d:\Amikom\Semester 6\Project Data Mining\Project\venv\lib\site-packages\xgboost\core.py:158: UserWarning: [16:05:00] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are no

      ✅ Selesai! Skor F1 Fold 1: 0.7598
   ▶️ Sedang melatih Fold ke-2...


d:\Amikom\Semester 6\Project Data Mining\Project\venv\lib\site-packages\xgboost\core.py:158: UserWarning: [16:05:00] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
d:\Amikom\Semester 6\Project Data Mining\Project\venv\lib\site-packages\xgboost\core.py:158: UserWarning: [16:05:01] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
d:\Amikom\Semester 6\Project Data Mining\Project\venv\lib\site-packages\xgboost\core.py:158: UserWarning: [16:05:01] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are no

      ✅ Selesai! Skor F1 Fold 2: 0.7652
   ▶️ Sedang melatih Fold ke-3...


...

      ✅ Selesai! Skor F1 Fold 3: 0.7502
   ▶️ Sedang melatih Fold ke-4...


d:\Amikom\Semester 6\Project Data Mining\Project\venv\lib\site-packages\xgboost\core.py:158: UserWarning: [16:05:03] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
d:\Amikom\Semester 6\Project Data Mining\Project\venv\lib\site-packages\xgboost\core.py:158: UserWarning: [16:05:03] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
d:\Amikom\Semester 6\Project Data Mining\Project\venv\lib\site-packages\xgboost\core.py:158: UserWarning: [16:05:03] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are no

      ✅ Selesai! Skor F1 Fold 4: 0.7622
   ▶️ Sedang melatih Fold ke-5...


d:\Amikom\Semester 6\Project Data Mining\Project\venv\lib\site-packages\xgboost\core.py:158: UserWarning: [16:05:04] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
d:\Amikom\Semester 6\Project Data Mining\Project\venv\lib\site-packages\xgboost\core.py:158: UserWarning: [16:05:04] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
d:\Amikom\Semester 6\Project Data Mining\Project\venv\lib\site-packages\xgboost\core.py:158: UserWarning: [16:05:04] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are no

      ✅ Selesai! Skor F1 Fold 5: 0.7806

🏆 KESIMPULAN 5-FOLD CROSS VALIDATION 🏆
Skor setiap Fold : [0.7598, 0.7652, 0.7502, 0.7622, 0.7806]
Rata-rata (Mean) : **0.7636**
Simpangan Baku   : ±0.0099 (Makin kecil makin stabil)
Penjelasan untuk Dosen: Model ini telah dievaluasi 5 kali berturut-turut pada potongan data yang berbeda, dan terbukti menghasilkan akurasi yang stabil. Modeling tidak hanya dilakukan satu kali.
